
# Notebook 2 - Final RAG Pipeline with HF LLM Query Rewriting + Reranking

This is the final Notebook 2 for your project.

Locked architecture:
1. Coordinator Agent
2. Retrieval Agent
3. Address Standardization Agent
4. Commute & Confidence Agent
5. Eligibility Agent
6. Escalation Agent

This notebook builds only the Retrieval Agent.

What it does:
- installs `uv`
- installs project packages with `uv pip`
- reads KB and employee data from BigQuery
- converts KB rows into LangChain Documents
- creates embeddings with Hugging Face
- builds a FAISS vector store
- uses a Hugging Face instruct LLM for query rewriting
- reranks retrieved documents
- supports batch retrieval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/address-intelligence-project"
os.makedirs(project_path, exist_ok=True)

folders = ["notebooks", "src", "data", "scripts", "docs"]

for f in folders:
    os.makedirs(os.path.join(project_path, f), exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



## Step 0: install uv

We use uv first because it handles package installs more cleanly than plain pip.
If your runtime is fresh, run this cell once.


In [ ]:

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"
!uv --version


downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.11.7 (x86_64-unknown-linux-gnu)



## Step 1: install project packages with uv

If Colab asks, restart runtime after this cell, then run the notebook from the top again.


In [ ]:

import os

os.environ["PATH"] += ":/root/.local/bin"

#!uv pip install --system   numpy==1.26.4   langchain   langchain-community   langchain-huggingface   faiss-cpu   sentence-transformers   transformers   accelerate   bitsandbytes   google-cloud-bigquery   db-dtypes   pandas   requests==2.32.4


In [ ]:
!pip uninstall -y numpy pandas faiss-cpu sentence-transformers transformers accelerate bitsandbytes langchain langchain-core langchain-community langchain-huggingface
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  langchain \
  langchain-community \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  google-cloud-bigquery \
  db-dtypes \
  requests==2.32.4

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: sentence-transformers 5.4.0
Uninstalling sentence-transformers-5.4.0:
  Successfully uninstalled sentence-transformers-5.4.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.2.28
Uninstalling langchain-core-1.2.28:
  Successfully uninstalled langchain-core-1.2.28
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 147.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions 

In [ ]:
import numpy as np
import pandas as pd
import torch

print(np.__version__)
print(pd.__version__)
print(torch.cuda.is_available())

1.26.4
2.2.2
True


In [ ]:

import json
import torch
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig



## Step 2: authenticate and connect to BigQuery


In [ ]:

auth.authenticate_user()
print("Authenticated successfully")

PROJECT_ID = "" #add your project id here
DATASET_ID = "address_intelligence_demo" #add your database id here
TABLE_EMPLOYEES = "employee_input"
TABLE_KB = "kb_documents"

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery project:", PROJECT_ID)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Authenticated successfully
Connected to BigQuery project: project-71bd54a3-0e82-43bd-a71
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB



## Step 3: read employee and KB data from BigQuery


In [ ]:

kb_query = f'''
SELECT doc_id, source_type, title, text
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_KB}`
ORDER BY doc_id
'''

employee_query = f'''
SELECT employee_id, employee_name, office_id, raw_home_address, office_address
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_EMPLOYEES}`
ORDER BY employee_id
'''

kb_df = client.query(kb_query).to_dataframe()
employee_df = client.query(employee_query).to_dataframe()

print("KB rows:", len(kb_df))
print("Employee rows:", len(employee_df))
kb_df.head(10)


KB rows: 10
Employee rows: 20


,doc_id,source_type,title,text
0,EX001,examples,Historical Correction Example 1,"Raw: 1818 Church Street, Nashvile, TN 37203. C..."
1,EX002,examples,Historical Correction Example 2,"Raw: 99 broadwai, nashville, tn 37201. Correct..."
2,EX003,examples,Historical Correction Example 3,Raw: 12 Music Sq W Nashville TN. Corrected: 12...
3,EX004,examples,Historical Correction Example 4,Raw: 123 Main Street Apartment Two Nashville T...
4,EX005,examples,Historical Correction Example 5,"Raw: 404 Not Found Ln, Brentwood, TN 37027. Co..."
5,OFF001,office,Nashville Main Office Metadata,Office ID 1 is Nashville Main Office located a...
6,POL001,policy,Commute Eligibility Policy,Employees are eligible for onsite assignment w...
7,POL002,policy,Manual Review Escalation Rules,Escalate cases when any of the following happe...
8,RULE001,rules,Address Standardization Rules,Standardize addresses using USPS-style convent...
9,RULE002,rules,Common Typo Corrections,"Examples: Broudway -> Broadway, broadwai -> Br..."



## Step 4: convert KB rows into LangChain Documents


In [ ]:

documents = []
for _, row in kb_df.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={
                "doc_id": row["doc_id"],
                "source_type": row["source_type"],
                "title": row["title"],
            }
        )
    )

print("Created LangChain documents:", len(documents))
documents[0]


Created LangChain documents: 10


Document(metadata={'doc_id': 'EX001', 'source_type': 'examples', 'title': 'Historical Correction Example 1'}, page_content='Raw: 1818 Church Street, Nashvile, TN 37203. Corrected: 1818 Church St, Nashville, TN 37203. Reason: corrected city typo and standardized street suffix.')


## Step 5: create embeddings and vector store


In [ ]:

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

VECTOR_DIR = "rag_vectorstore_final"
vectorstore.save_local(VECTOR_DIR)

print("Embedding model ready:", EMBEDDING_MODEL_NAME)
print("FAISS vector store created")
print("Saved vector store to:", VECTOR_DIR)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready: sentence-transformers/all-MiniLM-L6-v2
FAISS vector store created
Saved vector store to: rag_vectorstore_final



## Step 6: load a Hugging Face instruct LLM for query rewriting

Recommended model:
- Qwen/Qwen2.5-3B-Instruct

If needed, you can later swap to:
- microsoft/Phi-3.5-mini-instruct


In [ ]:

REWRITE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(REWRITE_MODEL_NAME)

rewrite_model = AutoModelForCausalLM.from_pretrained(
    REWRITE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)

rewrite_pipe = pipeline(
    "text-generation",
    model=rewrite_model,
    tokenizer=tokenizer,
    max_new_tokens=160,
    temperature=0.0,
    do_sample=False
)

print("LLM query rewriter ready:", REWRITE_MODEL_NAME)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM query rewriter ready: Qwen/Qwen2.5-3B-Instruct



## Step 7: create system prompt + user prompt for LLM query rewriting


In [ ]:

SYSTEM_PROMPT_QUERY_REWRITER = """
You are a retrieval query rewriting assistant for an enterprise address intelligence system.

Your job:
1. Read a messy employee-entered home address and an assigned office address.
2. Rewrite them into a clean, retrieval-friendly query.
3. Preserve important location details.
4. Expand the retrieval intent so the retriever can find:
   - commute eligibility policy
   - address standardization rules
   - typo correction examples
   - office metadata
   - manual review / escalation rules
5. Do not invent unsupported address fields.
6. Return only the rewritten retrieval query text.
"""

def build_user_prompt_for_rewriter(raw_home_address: str, office_address: str) -> str:
    return f"""
Raw employee home address:
{raw_home_address}

Assigned office address:
{office_address}

Rewrite this into a retrieval query that will help search the enterprise KB for the best supporting documents.
"""



## Step 8: build the HF LLM query rewriting function


In [ ]:

def rewrite_query_with_hf_llm(raw_home_address: str, office_address: str, text_generation_pipeline) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_QUERY_REWRITER},
        {"role": "user", "content": build_user_prompt_for_rewriter(raw_home_address, office_address)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = text_generation_pipeline(prompt)[0]["generated_text"]
    rewritten = output[len(prompt):].strip()
    return rewritten



## Step 9: test LLM query rewriting on one employee row


In [ ]:

sample_row = employee_df.iloc[2]
sample_row


,2
employee_id,E003
employee_name,Carla Smith
office_id,1
raw_home_address,"789 Broudway, Nashville, TN 37203"
office_address,"100 Healthcare Blvd, Nashville, TN 37205"


In [ ]:

raw_address = sample_row["raw_home_address"]
office_address = sample_row["office_address"]

rewritten_query = rewrite_query_with_hf_llm(
    raw_home_address=raw_address,
    office_address=office_address,
    text_generation_pipeline=rewrite_pipe
)

print("RAW ADDRESS:")
print(raw_address)
print("\nREWRITTEN QUERY:")
print(rewritten_query)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW ADDRESS:
789 Broudway, Nashville, TN 37203

REWRITTEN QUERY:
What are the commute eligibility policies, address standardization rules, and typo correction examples associated with the following addresses: 
- Home Address: 789 Broadway, Nashville, TN 37203
- Office Address: 100 Healthcare Blvd, Nashville, TN 37205
Please also include any relevant office metadata and manual review/escalation rules related to these addresses.



## Step 10: retrieve candidate docs from the vector store


In [ ]:

initial_docs = vectorstore.similarity_search(rewritten_query, k=6)

for i, doc in enumerate(initial_docs, start=1):
    print("=" * 80)
    print("Initial Rank:", i)
    print("Doc ID:", doc.metadata["doc_id"])
    print("Title:", doc.metadata["title"])
    print("Source type:", doc.metadata["source_type"])
    print("Text:", doc.page_content)


Initial Rank: 1
Doc ID: OFF001
Title: Nashville Main Office Metadata
Source type: office
Text: Office ID 1 is Nashville Main Office located at 100 Healthcare Blvd, Nashville, TN 37205. Commute checks for Office ID 1 should use this address as the destination.
Initial Rank: 2
Doc ID: RULE001
Title: Address Standardization Rules
Source type: rules
Text: Standardize addresses using USPS-style conventions where possible. Convert street suffixes to standard forms, normalize state names to two-letter abbreviations, and preserve apartment or suite numbers. Do not invent missing fields.
Initial Rank: 3
Doc ID: EX004
Title: Historical Correction Example 4
Source type: examples
Text: Raw: 123 Main Street Apartment Two Nashville Tennessee 37203. Corrected: 123 Main St Apt 2, Nashville, TN 37203. Reason: normalized state, apartment, punctuation, and suffix.
Initial Rank: 4
Doc ID: EX001
Title: Historical Correction Example 1
Source type: examples
Text: Raw: 1818 Church Street, Nashvile, TN 37203. 


## Step 11: rerank the candidate docs


In [ ]:

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Reranker ready:", RERANKER_MODEL_NAME)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker ready: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:

def rerank_documents(query_text, docs, reranker_model, top_k=4):
    pairs = [[query_text, d.page_content] for d in docs]
    scores = reranker_model.predict(pairs)

    scored = list(zip(docs, scores))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    final_docs = []
    for doc, score in scored[:top_k]:
        doc.metadata["rerank_score"] = float(score)
        final_docs.append(doc)
    return final_docs

reranked_docs = rerank_documents(
    query_text=rewritten_query,
    docs=initial_docs,
    reranker_model=reranker,
    top_k=4
)

for i, doc in enumerate(reranked_docs, start=1):
    print("=" * 80)
    print("Final Rank:", i)
    print("Doc ID:", doc.metadata["doc_id"])
    print("Title:", doc.metadata["title"])
    print("Rerank score:", round(doc.metadata["rerank_score"], 4))
    print("Text:", doc.page_content)


Final Rank: 1
Doc ID: RULE002
Title: Common Typo Corrections
Rerank score: -4.2123
Text: Examples: Broudway -> Broadway, broadwai -> Broadway, Nashvile -> Nashville, Murfresboro -> Murfreesboro, Peachtre -> Peachtree, Ter -> Terrace when context suggests a street suffix.
Final Rank: 2
Doc ID: OFF001
Title: Nashville Main Office Metadata
Rerank score: -4.759
Text: Office ID 1 is Nashville Main Office located at 100 Healthcare Blvd, Nashville, TN 37205. Commute checks for Office ID 1 should use this address as the destination.
Final Rank: 3
Doc ID: POL001
Title: Commute Eligibility Policy
Rerank score: -5.4832
Text: Employees are eligible for onsite assignment when their verified primary residence is within 60 miles driving distance of the assigned office. If the routing API fails or if the corrected address confidence is below 0.75, the case must be escalated for manual review.
Final Rank: 4
Doc ID: EX001
Title: Historical Correction Example 1
Rerank score: -5.5645
Text: Raw: 1818 Churc


## Step 12: package everything into the final Retrieval Agent function


In [ ]:

def retrieval_agent_final(
    raw_home_address,
    office_address,
    vectorstore,
    text_generation_pipeline,
    reranker_model,
    candidate_k=8,
    top_k=4
):
    rewritten_query = rewrite_query_with_hf_llm(
        raw_home_address=raw_home_address,
        office_address=office_address,
        text_generation_pipeline=text_generation_pipeline
    )

    candidate_docs = vectorstore.similarity_search(rewritten_query, k=candidate_k)
    final_docs = rerank_documents(rewritten_query, candidate_docs, reranker_model, top_k=top_k)

    return {
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "rewritten_query": rewritten_query,
        "retrieved_context": [
            {
                "doc_id": d.metadata["doc_id"],
                "title": d.metadata["title"],
                "source_type": d.metadata["source_type"],
                "rerank_score": d.metadata.get("rerank_score"),
                "text": d.page_content
            }
            for d in final_docs
        ]
    }

retrieval_output = retrieval_agent_final(
    raw_home_address=raw_address,
    office_address=office_address,
    vectorstore=vectorstore,
    text_generation_pipeline=rewrite_pipe,
    reranker_model=reranker,
    candidate_k=8,
    top_k=4
)

retrieval_output


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'raw_home_address': '789 Broudway, Nashville, TN 37203',
 'office_address': '100 Healthcare Blvd, Nashville, TN 37205',
 'rewritten_query': 'What are the commute eligibility policies, address standardization rules, and typo correction examples associated with the following addresses: \n- Home Address: 789 Broadway, Nashville, TN 37203\n- Office Address: 100 Healthcare Blvd, Nashville, TN 37205\nPlease also include any relevant office metadata and manual review/escalation rules related to these addresses.',
 'retrieved_context': [{'doc_id': 'RULE002',
   'title': 'Common Typo Corrections',
   'source_type': 'rules',
   'rerank_score': -4.2122955322265625,
   'text': 'Examples: Broudway -> Broadway, broadwai -> Broadway, Nashvile -> Nashville, Murfresboro -> Murfreesboro, Peachtre -> Peachtree, Ter -> Terrace when context suggests a street suffix.'},
  {'doc_id': 'OFF001',
   'title': 'Nashville Main Office Metadata',
   'source_type': 'office',
   'rerank_score': -4.758999347686768,
  


## Step 13: batch retrieval for many employees


In [ ]:

batch_outputs = []

for _, row in employee_df.head(5).iterrows():
    result = retrieval_agent_final(
        raw_home_address=row["raw_home_address"],
        office_address=row["office_address"],
        vectorstore=vectorstore,
        text_generation_pipeline=rewrite_pipe,
        reranker_model=reranker,
        candidate_k=8,
        top_k=4
    )

    batch_outputs.append({
        "employee_id": row["employee_id"],
        "employee_name": row["employee_name"],
        "raw_home_address": row["raw_home_address"],
        "rewritten_query": result["rewritten_query"],
        "top_doc_titles": [x["title"] for x in result["retrieved_context"]]
    })

batch_df = pd.DataFrame(batch_outputs)
batch_df


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,employee_id,employee_name,raw_home_address,rewritten_query,top_doc_titles
0,E001,Alice Johnson,"123 Main St Apt 2, Nashville, TN 37203","What are the commute eligibility policies, add...","[Historical Correction Example 4, Nashville Ma..."
1,E002,Brian Lee,"456 elm street, franklin tn 37064","What are the commute eligibility policies, add...","[Nashville Main Office Metadata, Commute Eligi..."
2,E003,Carla Smith,"789 Broudway, Nashville, TN 37203","What are the commute eligibility policies, add...","[Common Typo Corrections, Nashville Main Offic..."
3,E004,David Patel,"1600 West End Ave Unit 5, Nashville, TN 37203","What are the commute eligibility policies, add...","[Nashville Main Office Metadata, Commute Eligi..."
4,E005,Eva Brown,"22 Peachtre St NW, Atlanta, GA 30303","What are the commute eligibility policies, add...","[Common Typo Corrections, Historical Correctio..."


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



## Step 14: save retrieval artifacts locally for later notebooks


In [ ]:

with open("retrieval_final_sample_output.json", "w") as f:
    json.dump(retrieval_output, f, indent=2)

batch_df.to_csv("retrieval_final_batch_preview.csv", index=False)

print("Saved files:")
print("- retrieval_final_sample_output.json")
print("- retrieval_final_batch_preview.csv")
print("- rag_vectorstore_final/")


Saved files:
- retrieval_final_sample_output.json
- retrieval_final_batch_preview.csv
- rag_vectorstore_final/



## Finished

This file includes:
- BigQuery KB loading
- LangChain Documents
- Hugging Face embeddings
- FAISS vector search
- HF LLM-based query rewriting with system + user prompts
- reranking
- batch retrieval


Creating helper functions (optional)

In [4]:


from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/address-intelligence-project"
os.listdir("/content/drive/MyDrive/address-intelligence-project")
os.listdir("/content/drive/MyDrive/address-intelligence-project/src")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[]

In [5]:
%%writefile /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py

def test_func():
    return "hello"

Writing /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py


In [6]:
os.listdir("/content/drive/MyDrive/address-intelligence-project/src")

['retrieval_helper.py']

In [7]:
%%writefile /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

def init_retrieval_components(documents):
    embeddings = HuggingFaceEmbeddings()
    vectorstore = FAISS.from_documents(documents, embeddings)
    return vectorstore

def retrieval_agent(query, vectorstore, k=3):
    return vectorstore.similarity_search(query, k=k)

Overwriting /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py
